<a href="https://colab.research.google.com/github/sdoucette9/Colab/blob/main/CS399_Team_3_Tabular_Classification_Example.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Install tabstar package and the version of HF transformers that it needs

This example uses the TabSTAR model from HuggingFace, documented here: https://huggingface.co/alana89/TabSTAR

In [ ]:
!pip install transformers==4.49.0
!pip install tabstar

# Load training and testing data

Note:  This will not work if you have not yet uploaded these data files into your Colab workspace.  Click the Files icon on the left and upload these example files.  I obtained them from this Kaggle posting: https://www.kaggle.com/datasets/coderanand/university-query-priority-classification

In [ ]:
import pandas as pd
df = pd.read_csv('/CS399 ER Wait Time Dataset.csv')
df = df[df.columns[:18]]
# I only want up to Bottleneck Risk Level which is column 18

# Since my dataset is not split into training and testing, I will manually split it into 2 seperate dataframes
split_index = int(len(df)*0.80) #Training with be 80% of the orginal dataset & Testing will be 20%

df_training = df[:split_index]
df_testing = df[split_index:]

df_training


,Visit ID,Patient ID,Hospital ID,Hospital Name,Region,Visit Date,Day of Week,Season,Time of Day,Urgency Level,Nurse-to-Patient Ratio,Specialist Availability,Facility Size (Beds),Time to Registration (min),Time to Triage (min),Time to Medical Professional (min),Total Wait Time (min),Bottleneck Risk Level
0,HOSP-1-20240210-0001,PAT-00001,HOSP-1,Springfield General Hospital,Urban,2/10/2024 20:20,Saturday,Winter,Late Morning,Medium,4,3,92,17,22,66,105,Medium
1,HOSP-3-20241128-0001,PAT-00002,HOSP-3,Northside Community Hospital,Rural,11/28/2024 2:07,Thursday,Fall,Evening,Medium,4,0,38,9,30,30,69,Medium
2,HOSP-3-20240930-0002,PAT-00003,HOSP-3,Northside Community Hospital,Rural,9/30/2024 4:02,Monday,Fall,Evening,Low,5,1,38,38,40,125,203,High
3,HOSP-2-20240227-0001,PAT-00004,HOSP-2,Riverside Medical Center,Urban,2/27/2024 0:31,Tuesday,Winter,Evening,High,4,5,94,8,16,64,88,Medium
4,HOSP-1-20240306-0002,PAT-00005,HOSP-1,Springfield General Hospital,Urban,3/6/2024 16:52,Wednesday,Spring,Afternoon,Low,4,8,74,26,29,63,118,Medium
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3995,HOSP-3-20240203-0790,PAT-03996,HOSP-3,Northside Community Hospital,Rural,2/3/2024 21:21,Saturday,Winter,Evening,Critical,2,0,29,1,7,26,34,Low
3996,HOSP-4-20240218-0798,PAT-03997,HOSP-4,St. Mary’s Regional Health,Rural,2/18/2024 10:57,Sunday,Winter,Afternoon,Low,4,2,29,17,41,104,162,High
3997,HOSP-5-20240329-0792,PAT-03998,HOSP-5,Summit Health Center,Urban,3/29/2024 19:25,Friday,Spring,Afternoon,Low,5,7,69,20,48,89,157,High
3998,HOSP-1-20240725-0810,PAT-03999,HOSP-1,Springfield General Hospital,Urban,7/25/2024 18:45,Thursday,Summer,Afternoon,Low,4,8,108,19,71,108,198,High


# Separate inputs from outputs

We want to predict the final column from the others.  So we have to separate them.  Here I'm working with just the training data.

In [ ]:

inputs = df_training[df_training.columns[:-1]]  # all but last col
outputs = df_training[df_training.columns[-1]]  # last col
inputs.shape, outputs.shape

((4000, 17), (4000,))

# Train model

TabSTAR models have been pre-trained, but now we are fine-tuning them on the specific data we care about.  This takes a little while, as you can see from the extensive output.

In [ ]:
from tabstar.tabstar_model import TabSTARClassifier
tabstar = TabSTARClassifier()
tabstar.fit(inputs, outputs)

❗ Warning: 8 CPU devices requested, but only 2 available.
🖥️ Using device: cuda:0


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

config.json:   0%|          | 0.00/535 [00:00<?, ?B/s]

.DS_Store:   0%|          | 0.00/6.15k [00:00<?, ?B/s]

...


Epochs:  16%|█▌        | 8/50 [01:16<06:41,  9.57s/it]

Epoch 8 || Train 0.0330 || Val 0.0250 || Metric 0.9997  😓 [4 / 5]



Batches:  98%|█████████▊| 56/57 [00:08<00:00,  6.40it/s]
                                                      

Epoch 9 || Train 0.0376 || Val 0.0461 || Metric 0.9997  😓 [5 / 5]
🛑 Early stopping at epoch 9
🏆 Best checkpoint: Epoch 8 with loss 0.0250
Threshold: 0.0473 was chosen for best val loss of 0.0250
📊 Averaging 5 checkpoints:
- checkpoint_epoch_4.pt (val_loss=0.0306)
- checkpoint_epoch_5.pt (val_loss=0.0277)
- checkpoint_epoch_6.pt (val_loss=0.0349)
- checkpoint_epoch_7.pt (val_loss=0.0471)
- checkpoint_epoch_8.pt (val_loss=0.0250)
💾 Saved averaged checkpoint to .tabstar_checkpoint/20260316_181557/checkpoint_averaged.pt


✅ Saved averaged model to .tabstar_checkpoint/20260316_181557/averaged_model
📈 Averaged checkpoint || Val Loss: 0.0195 || Val Metric: 0.9999


# Separate inputs and outputs for testing

Same as before, but now the testing data instead of the training data.

In [ ]:
inputs = df_testing[df_testing.columns[:-1]]  # all but last col
outputs = df_testing[df_testing.columns[-1]]  # last col
inputs.shape, outputs.shape

((1000, 17), (1000,))

# Test model

In [ ]:
from sklearn.metrics import classification_report
predictions = tabstar.predict(inputs)
print(classification_report(outputs, predictions))
# Attaches predictions back to test data

df_testing = df_testing.copy()
df_testing["Predicted_Risk"] = predictions

# Counts all the risk levels per hospital

risk_summary = (
    df_testing
    .groupby(["Hospital Name", "Predicted_Risk"])
    .size()
    .unstack(fill_value=0)
)

print("\nRisk Levels Per Hospital:\n")
print(risk_summary)

              precision    recall  f1-score   support

        High       1.00      0.99      0.99       275
         Low       1.00      0.99      1.00       496
      Medium       0.97      1.00      0.98       229

    accuracy                           0.99      1000
   macro avg       0.99      0.99      0.99      1000
weighted avg       0.99      0.99      0.99      1000


Risk Levels Per Hospital:

Predicted_Risk                High  Low  Medium
Hospital Name                                  
Northside Community Hospital    65  102      42
Riverside Medical Center        62   98      54
Springfield General Hospital    54   91      39
St. Mary’s Regional Health      38  102      57
Summit Health Center            53   99      44


# Save the model

You can also save the model after you've trained it, so that it's ready to just load and use in your app.  In other words, you train it as part of development, save it, then just load it and use it in the deployed version of the app.

In [ ]:
tabstar.save('/content/model.pkl') # Python saves data in "pickle" files

# Clear the whole model out of memory and GPU

(You don't need this code, because your training and usage will be in different Python contexts entirely.  I need it here because this example is all in one context.)

In [ ]:
# Delete model from memory
del tabstar
# Run garbage collection to be sure it's really gone
import gc
gc.collect()
# Clear GPU memory as well
import torch
torch.cuda.empty_cache()

# How to load and re-use

In [ ]:
from tabstar.tabstar_model import TabSTARClassifier
reloaded = TabSTARClassifier.load('/content/model.pkl')
reloaded.predict(inputs) # check to be sure the loaded model can be run

array(['Low', 'Low', 'High', 'High', 'High', 'Medium', 'High', 'High',
       'Low', 'High', 'Low', 'Low', 'Low', 'Low', 'Low', 'Medium',
       'Medium', 'Low', 'Low', 'High', 'Medium', 'Low', 'Low', 'High',
       'Medium', 'Medium', 'High', 'Medium', 'High', 'Low', 'Low', 'Low',
       'Low', 'Low', 'Low', 'High', 'Medium', 'Low', 'Low', 'Low',
       'Medium', 'Low', 'High', 'Low', 'Low', 'Low', 'High', 'Low',
       'Medium', 'Low', 'High', 'High', 'Low', 'High', 'Low', 'High',
       'Low', 'Low', 'Low', 'Low', 'Low', 'Low', 'Medium', 'High',
       'Medium', 'Medium', 'Medium', 'Low', 'Low', 'Low', 'Low', 'High',
       'Low', 'Medium', 'Low', 'Low', 'Low', 'High', 'Low', 'High',
       'Medium', 'Low', 'High', 'Medium', 'Low', 'Low', 'Low', 'High',
       'High', 'Low', 'Medium', 'Low', 'Low', 'Medium', 'Low', 'Low',
       'Low', 'Low', 'Medium', 'High', 'Medium', 'Low', 'Medium', 'High',
       'Low', 'High', 'High', 'Medium', 'Medium', 'Medium', 'High',
       'Medium', 'Low